In [1]:
import os, sys
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

c:\Marco Conti\Projetos\mais_einstein\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_tempetatura = spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_temperatura.parquet")
df_tempetatura.printSchema()
df_tempetatura.limit(10).show()

root
 |-- data_medicao: date (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+------------+--------+---------+-----------+------------------+--------------+
|data_medicao|latitude|longitude|  indicador|             valor|unidade_medida|
+------------+--------+---------+-----------+------------------+--------------+
|  2025-01-01|     6.0|    -74.0|temperatura|21.317346191406273|       celsius|
|  2025-01-01|     6.0|   -73.75|temperatura|17.715447998046898|       celsius|
|  2025-01-01|     6.0|    -73.5|temperatura|16.692193603515648|       celsius|
|  2025-01-01|     6.0|   -73.25|temperatura|15.932000732421898|       celsius|
|  2025-01-01|     6.0|    -73.0|temperatura|13.354058837890648|       celsius|
|  2025-01-01|     6.0|   -72.75|temperatura|11.741571044921898|       celsius|
|  2025-01-01|     6.0|    

Obtem as estatísticas de temperatura por Ano e Mês:
- Temperatura mínima
- Temperatura máxima
- Temperatura média
- Percentil 5%
- Percentil 90%

In [4]:
df_base = (
    df_tempetatura
        .withColumn("ano", F.year("data_medicao"))
        .withColumn("mes", F.month("data_medicao"))
)

df_stats = (
    df_base
    .groupBy(
        "ano",
        "mes",
        "latitude",
        "longitude"
    )
    .agg(
        F.min("valor").alias("temp_min_mes"),
        F.max("valor").alias("temp_max_mes"),
        F.avg("valor").alias("temp_media_mes"),
        F.expr("percentile_approx(valor, 0.05)").alias("percentil_05_mes"),
        F.expr("percentile_approx(valor, 0.90)").alias("percentil_90_mes")
    )
)

df_stats.printSchema()
df_stats.show(10, truncate=False)

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)
 |-- percentil_05_mes: double (nullable = true)
 |-- percentil_90_mes: double (nullable = true)

+----+---+--------+---------+------------------+------------------+------------------+------------------+------------------+
|ano |mes|latitude|longitude|temp_min_mes      |temp_max_mes      |temp_media_mes    |percentil_05_mes  |percentil_90_mes  |
+----+---+--------+---------+------------------+------------------+------------------+------------------+------------------+
|2025|1  |-34.0   |-73.75   |14.811090087890648|18.047540283203148|16.617592891570084|15.427606201171898|17.296075439453148|
|2025|1  |-34.0   |-72.75   |15.337213134765648|17.719506835937523|16.555147035660305|15.579064941406273|17.040

In [5]:
# Anexar dados estatíscos e percentis ao dado diário
# Será utilizados para calcular as ondas de calor (periodos de dias consecutivos)
df_dia = (
    df_base.join(
        df_stats.select(
            "ano",
            "mes",
            "latitude",
            "longitude",
            "temp_min_mes",
            "temp_max_mes",
            "temp_media_mes",
            "percentil_05_mes",
            "percentil_90_mes"
        ),
        [
            "ano",
            "mes",
            "latitude",
            "longitude"
        ],
        how="left"
    )
)

print("Número de registros no DataFrame diário:", df_dia.count()) # 151662
df_dia.printSchema()
df_dia.show(10, truncate=False)

Número de registros no DataFrame diário: 783587
root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)
 |-- percentil_05_mes: double (nullable = true)
 |-- percentil_90_mes: double (nullable = true)

+----+---+--------+---------+------------+-----------+------------------+--------------+------------------+------------------+------------------+------------------+------------------+
|ano |mes|latitude|longitude|data_medicao|indicador  |valor             |unidade_medida|temp_min_mes      |temp_max_mes      |temp_media_mes    |percentil_05_mes  |percentil_90_mes  |
+----+---+--------+---------+

In [6]:
# Classificar extremos
df_dia = (
    df_dia
    .withColumn(
        "extremo_alto",
        F.when(F.col("valor") > F.col("percentil_90_mes")
              ,(F.col("valor") - F.col("percentil_90_mes"))).otherwise(0))
    .withColumn(
        "extremo_baixo",
        F.when(F.col("valor") < F.col("percentil_05_mes")
              ,(F.col("percentil_05_mes") - F.col("valor"))).otherwise(0)
    )
)

df_dia.printSchema()
df_dia.filter("extremo_baixo != 0 or extremo_alto != 0").show(10, truncate=False)
# (df_dia
#     .select('data_medicao', 'latitude', 'longitude', 'percentil_05_mes', 'valor', 'percentil_90_mes'
#            ,'extremo_alto', 'extremo_baixo' )
#     .orderBy("latitude", "longitude", "data_medicao")
#     .show(1000, truncate=False))

root
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)
 |-- temp_min_mes: double (nullable = true)
 |-- temp_max_mes: double (nullable = true)
 |-- temp_media_mes: double (nullable = true)
 |-- percentil_05_mes: double (nullable = true)
 |-- percentil_90_mes: double (nullable = true)
 |-- extremo_alto: double (nullable = true)
 |-- extremo_baixo: double (nullable = true)

+----+---+--------+---------+------------+-----------+------------------+--------------+------------------+------------------+------------------+------------------+------------------+-----------------+-------------+
|ano |mes|latitude|longitude|data_medicao|indicador  |valor             |unidade_medida|temp_min_mes      |temp_max_mes      |temp_media_mes

In [7]:
# Identifica os dias quentes:

# Considera-se dia quente quando a temperatura excede o percentil 90 do mês

df_flag = df_dia.withColumn(
    "flag_dia_quente",
    F.when(F.col("valor") > F.col("percentil_90_mes"), 1).otherwise(0)
)

# Mantém apenas os dias que atenderam ao critério
df_quentes = df_flag.filter(F.col("flag_dia_quente") == 1)


# Janela ordenada por data para cada ponto geográfico
janela_loc = Window.partitionBy("latitude", "longitude").orderBy("data_medicao")

# Ao subtrair a ordem do registro (rn) da data, dias consecutivos geram
# exatamente o mesmo identificador de grupo (grupo_id)
df_eventos = df_quentes \
    .withColumn("rn", F.row_number().over(janela_loc)) \
    .withColumn("grupo_id", F.expr("date_sub(data_medicao, CAST(rn AS INT))"))


In [8]:
# 1. Identificação dos Eventos e Validação da Duração (Global, sem quebra de mês/ano)
df_eventos_duracao = (
    df_eventos
    .groupBy("latitude", "longitude", "grupo_id")
    .agg(
        F.count("data_medicao").alias("duracao_total_onda"),
        F.min("data_medicao").alias("inicio_onda"),
        F.max("data_medicao").alias("fim_onda")
    )
    .filter(F.col("duracao_total_onda") > 2) # Filtra apenas eventos reais (> 2 dias)
)

# 2. Retornar os DIAS INDIVIDUAIS das ondas válidas mantendo os metadados da onda
df_dias_em_onda = (
    df_eventos
    .join(df_eventos_duracao, ["latitude", "longitude", "grupo_id"], "inner")
    .select(
        "latitude", 
        "longitude", 
        "data_medicao", 
        "ano", 
        "mes", 
        "grupo_id", 
        "duracao_total_onda"
    )
)

In [9]:
df_metricas_mensal = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano", "mes")
    .agg(
        # Número de ondas distintas que passaram/ocorreram neste mês específico
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # Total exato de dias sob onda de calor DENTRO deste mês
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # Duração total dos eventos que afetaram este mês (máxima e média)
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas")
    )
    .orderBy("ano", "mes", "latitude", "longitude")
)

df_metricas_mensal.show(10, truncate=False)

+--------+---------+----+---+------------------+--------------------------+-------------------+-------------------+
|latitude|longitude|ano |mes|numero_ondas_calor|frequencia_dias_onda_calor|duracao_maxima_onda|duracao_media_ondas|
+--------+---------+----+---+------------------+--------------------------+-------------------+-------------------+
|-34.0   |-72.0    |2025|1  |1                 |3                         |3                  |3.0                |
|-34.0   |-70.0    |2025|1  |1                 |3                         |3                  |3.0                |
|-34.0   |-69.75   |2025|1  |1                 |3                         |3                  |3.0                |
|-34.0   |-66.0    |2025|1  |1                 |3                         |3                  |3.0                |
|-34.0   |-65.75   |2025|1  |1                 |3                         |3                  |3.0                |
|-34.0   |-65.5    |2025|1  |1                 |3                       

In [10]:
df_metricas_anual = (
    df_dias_em_onda
    .groupBy("latitude", "longitude", "ano")
    .agg(
        # Número de ondas distintas que tiveram ao menos um dia neste ano
        F.countDistinct("grupo_id").alias("numero_ondas_calor"),
        
        # Total exato de dias do ano passados em onda de calor
        F.count("data_medicao").alias("frequencia_dias_onda_calor"),
        
        # Duração máxima e média dos eventos ocorridos no ano
        F.max("duracao_total_onda").alias("duracao_maxima_onda"),
        F.round(F.avg("duracao_total_onda"), 2).alias("duracao_media_ondas")
    )
    .orderBy("ano", "latitude", "longitude")
)

df_metricas_anual.show(10, truncate=False)

+--------+---------+----+------------------+--------------------------+-------------------+-------------------+
|latitude|longitude|ano |numero_ondas_calor|frequencia_dias_onda_calor|duracao_maxima_onda|duracao_media_ondas|
+--------+---------+----+------------------+--------------------------+-------------------+-------------------+
|-34.0   |-72.0    |2025|1                 |3                         |3                  |3.0                |
|-34.0   |-70.0    |2025|1                 |3                         |3                  |3.0                |
|-34.0   |-69.75   |2025|1                 |3                         |3                  |3.0                |
|-34.0   |-66.0    |2025|1                 |3                         |3                  |3.0                |
|-34.0   |-65.75   |2025|1                 |3                         |3                  |3.0                |
|-34.0   |-65.5    |2025|1                 |3                         |3                  |3.0          